In [ ]:
import numpy as np
import anndata as an
import scanpy as sc
import scanpy.external as sce
import scipy
import os
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn.functional as F
import pandas as pd
import PyComplexHeatmap as pch
from scipy.stats import fisher_exact
from statsmodels.stats.multitest import multipletests
import PyComplexHeatmap as pch
import pyreadr
import pickle

In [ ]:
qs = torch.load('results/spatial/qs.pt',weights_only = False)
h  = torch.load('results/spatial/h.pt',weights_only = False)
z  = torch.load('results/spatial/zs.pt',weights_only = False)
Cs  = torch.load('results/spatial/Cs.pt',weights_only = False)

In [ ]:
GCBC,NBC_MBC,PC,CD4T = [np.argmax(q, axis=1) for q in qs]

GCBC_label, NBC_MBC_label, PC_label, CD4T_label = [
    [str(x) for x in np.argmax(q.detach().numpy(), axis=1)] for q in qs
]
qs = [x.detach().numpy() for x in qs]
labels = [GCBC_label, NBC_MBC_label, PC_label,CD4T_label]

In [ ]:
adata = an.AnnData(h[0].detach().numpy())
#adata.obs['confidence'] = qs[3][np.arange(len(qs[3])), qs[3].argmax(1)]
adata.obs['cluster'] = GCBC_label
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata,color=['cluster'],title='UMAP of epithelial Clusters in Latent Space')

In [ ]:
pairs = [(0,1), (1,2), (2,3),(3,0), (0,2),(1,3)]
def normalize(A, B):
    C = (A + B.T) / 2
    return (C - C.min()) / (C.max() - C.min())
C = [normalize(Cs[i][j], Cs[j][i]) for i,j in pairs]
C_new = []
for c in C:
    c_new = np.where(c > 0.75, c, 0) 
    C_new.append(c_new)
    

In [ ]:
plt.figure(figsize=(8,6))

sns.heatmap(C[0], fmt="g", cmap='Blues') 
plt.title("Matrix Heatmap")
plt.xlabel("Columns")
plt.ylabel("Rows")
plt.show()

In [ ]:
g = torch.load('data/spatial/graph/intra/CD4_T.pt',weights_only = False)
df = pd.DataFrame(CD4T_label, columns=['cluster'])
order = df.sort_values('cluster').index.tolist()

data_sorted = pd.DataFrame(g).iloc[order, order].reset_index(drop=True)
data_sorted.columns = range(len(data_sorted.columns)) 
df_sorted = df.iloc[order].reset_index(drop=True)

row_ha = pch.HeatmapAnnotation(
    cluster=pch.anno_simple(df_sorted.cluster, add_text=True, legend=True, cmap='tab20'),
    axis=0
)
col_ha = pch.HeatmapAnnotation(
    cluster=pch.anno_simple(df_sorted.cluster, add_text=True, legend=True, cmap='tab20'),
    axis=1
)

plt.figure(figsize=(7, 6))
cm = pch.ClusterMapPlotter(
    data=data_sorted,
    top_annotation=col_ha,
    left_annotation=row_ha,
    row_cluster=False,
    col_cluster=False,
    cmap='Reds',
    vmin=0,
    vmax=1,
    row_names_side='left',
    rasterized=True,
    label='AUC'
)
plt.show()

In [ ]:
path = "data/spatial/selected_genes/BCLL-8-T.csv"
cts = ['GCBC','NBC_MBC','PC''CD4_T'] 
df = pd.read_csv(path,index_col=0)
GCBC = df['GCBC'].dropna()
NBC_MBC = df['NBC_MBC'].dropna()
PC = df['PC'].dropna()
CD4T = df['CD4_T'].dropna()

In [ ]:
coor = pd.read_csv("data/spatial/coordinates.csv",index_col=0)
stged_ct1 = pd.read_csv(f'data/test/stged/epithelial.csv',index_col=0)
spots = stged_ct1.columns
pos = (
    coor.loc[coor.index.intersection(spots)]
        #.loc[spots][["row", "col"]]
)[["row", "col"]]

pos = pos.values
pos

In [ ]:
pairs = [(0,1), (1,2), (2,3),(3,0), (0,2),(1,3)]
mat = pd.DataFrame(np.zeros((20,20)))
i = 0
for (x,y) in pairs:
    
    row = x*5
    col = y*5
    mat.iloc[row:row+5, col:col+5] = np.array(C_new[i])
    i = i+1

#for i in range(6): 
    #row = i * 6
    #col = i * 6
    #mat.iloc[row:row+6, col:col+6] = np.identity(6)

mat = mat.to_numpy()
mat = np.maximum(mat, mat.T)

In [ ]:
df = pd.DataFrame(['GCBC']*5+['NBC_MBC']*5+['PC']*5+['CD4T']*5,columns=['cell_type'])
df['cluster'] =[0,1,2,3,4]*4

col_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster = pch.anno_simple(df.cluster, add_text =False, legend=True, cmap='tab20'), 
    axis=1
)
plt.figure(figsize=(10.5, 9))
cm = pch.ClusterMapPlotter(
    data=pd.DataFrame(torch.vstack(list(z)).detach().numpy()).T,
    top_annotation=col_ha,
    #left_annotation=row_ha,
    #row_split=df.cluster,
    #row_split=6,
    col_cluster=True,
    col_dendrogram=True, 
    #row_dendrogram_size=2.5,
    #col_cluster=True,
    cmap='Reds',
    vmin=0,
    vmax=1,
    row_names_side='left',
    rasterized=True,
    label='AUC'
)
plt.show()

In [ ]:
df = pd.DataFrame(['GCBC']*5+['NBC_MBC']*5+['PC']*5+['CD4T']*5,columns=['cell_type'])
df['cluster'] =[0,1,2,3,4]*4
row_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster = pch.anno_simple(df.cluster, add_text =False, legend=True, cmap='tab20'),
    axis=0
)
col_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster = pch.anno_simple(df.cluster, add_text =False, legend=True, cmap='tab20'),
    axis=1
)
plt.figure(figsize=(14, 12))
cm = pch.ClusterMapPlotter(
    data=pd.DataFrame(mat),
    top_annotation=col_ha,
    left_annotation=row_ha,
    row_split=df.cell_type,
    col_split=df.cell_type, 
    row_cluster=False,
    col_cluster=False,
    cmap='Reds',
    vmin=0,
    vmax=mat.max(),
    row_names_side='left',
    rasterized=False,
    linewidths=0,
    #annot=pd.DataFrame(sig_matrix),
    #fmt='',# <-- pass sig_matrix here
    #annot_kws={"color":"blue", "fontsize":20, "fontweight":"bold"},
)

In [ ]:
adata = sc.read_h5ad('data/test/tonsil_spatial.h5ad')
adata = adata[adata.obs['donor_id']=='BCLL-8-T']
adata = adata[spots,:].copy()
dot = pd.read_csv("data/test/dot.csv", index_col=0)
dot = dot.loc[coor.index.intersection(spots)]
adata.obsm["proportion"] = dot.values
adata.uns["cell_type"] = dot.columns.tolist()

In [ ]:
stged_ct1 = pd.read_csv(f'data/test/stged/GCBC.csv', index_col=0)
stged_ct2 = pd.read_csv(f'data/test/stged/NBC_MBC.csv', index_col=0)
stged_ct3 = pd.read_csv(f'data/test/stged/CD4_T.csv', index_col=0)

c = np.array(GCBC.iloc[np.where(np.array(GCBC_label) == '4')[0]])
d = np.array(NBC_MBC.iloc[np.where(np.array(NBC_MBC_label) == '0')[0]])
e = np.array(CD4T.iloc[np.where(np.array(CD4T_label) == '0')[0]])

mean1 = (stged_ct1.loc[c]).mean(axis=0).values
#mean1 = adata[:,c].X.mean(axis=1)
    
#
mean2 = (stged_ct2.loc[d]).mean(axis=0).values
#mean2 = adata[:,d].X.mean(axis=1)

mean3 = (stged_ct3.loc[e]).mean(axis=0).values
#e = np.intersect1d(c,d)

print(c)
print(d)
print(e)

In [ ]:
cell_types = adata.uns["cell_type"]
idx1 = cell_types.index('GCBC')
prop1 = adata.obsm["proportion"][:, idx1]
prop1[prop1 < 0.05] = 0
#prop1[prop1 >= 0.05] = 1

idx2 = cell_types.index('NBC_MBC')
prop2 = adata.obsm["proportion"][:, idx2]
prop2[prop2 < 0.05] = 0
#prop2[prop2 >= 0.05] = 1

idx3 = cell_types.index('CD4_T')
prop3 = adata.obsm["proportion"][:, idx3]
prop3[prop3 < 0.05] = 0


#exp = (adata[:, c].X)#.multiply(prop[:, np.newaxis])
#score = exp1.mean(axis=1)
adata.obs["score1"] = mean1 * prop1
adata.obs["score2"] = mean2 * prop2
adata.obs["score3"] = mean3 * prop3
adata.obs['joint'] = mean1 * mean2# * prop2 * prop2


sc.pl.embedding(
    adata,
    color="score1",
    basis="spatial_rot",
    cmap="Reds",
    size=100
)
sc.pl.embedding(
    adata,
    color="score2",
    basis="spatial_rot",
    cmap="Blues",
    size=100
)
sc.pl.embedding(
    adata,
    color="score3",
    basis="spatial_rot",
    cmap="Greens",
    size=100
)

In [ ]:
#e = np.array(GCBC.iloc[np.where(np.array(GCBC_label) == '2')[0]])
e = np.array(CD4T)
for i in e:
    #e = np.array(GCBC.iloc[np.where(np.array(GCBC_label) == f'{i}')[0]])
    print(i)

In [ ]:


z_all = torch.vstack(list(z)).detach().numpy()  # shape: (total_clusters, d)

adata = an.AnnData(X=z_all)

cell_types = ['GCBC']*5 + ['NBC_MBC']*5 + ['PC']*5 + ['CD4T']*5
adata.obs['cell_type'] = pd.Categorical(cell_types)
adata.obs['cluster'] = [str(i % 5) for i in range(z_all.shape[0])]

sc.pp.neighbors(adata, use_rep='X')
sc.tl.umap(adata)

sc.pl.umap(adata, color=['cell_type', 'cluster'], ncols=2)

In [ ]:
import PyComplexHeatmap as pch

z_all = torch.vstack(list(z)).detach().numpy()  # shape: (20, d)

df = pd.DataFrame({
    'cell_type': ['GCBC']*5 + ['NBC_MBC']*5 + ['PC']*5 + ['CD4T']*5,
    'cluster': [str(i % 5) for i in range(20)]
})

row_ha = pch.HeatmapAnnotation(
    cell_type=pch.anno_simple(df.cell_type, add_text=False, legend=True, cmap='tab20'),
    cluster=pch.anno_simple(df.cluster, add_text=False, legend=True, cmap='Set2'),
    axis=0
)

plt.figure(figsize=(8, 6))
cm = pch.ClusterMapPlotter(
    data=pd.DataFrame(z_all),
    left_annotation=row_ha,
    row_cluster=True,
    col_cluster=True,
    row_dendrogram=True,
    cmap='RdBu_r',
    row_names_side='right',
    rasterized=True,
    label='Z value'
)
plt.show()